# Deeper Networks

Every network we've built so far had **one** hidden layer: the XOR network, and the `hidden_layer_sizes=(h,)` networks of Lab 7. Real networks stack many. This notebook asks what a second (or tenth) hidden layer adds, and what the *sizes* of the layers do:

- a layer **wider** than its input gives the data room to come apart;
- a layer **narrower** than its input forces the network to summarize, and *where* that narrow layer sits matters a lot.

These two moves, widen and narrow, are inside every modern network, including the transformers behind large language models.

### By the end, you'll be able to

1. Read a network with two or more hidden layers: its picture, its notation, its weight shapes, and its parameter count. Write the forward pass as a loop over layers.
2. Explain what widening a layer does: with more hidden units than inputs, h-space has more dimensions, and data that no line separates in the original features can become separable by a plane.
3. Explain what a narrow layer (a **bottleneck**) does: it forces the network to summarize. Explain why the same 2-unit bottleneck does far better after a wide layer than before one.
4. Use the terms **up-projection** and **down-projection** for a widening and a narrowing layer, and recognize the expand-then-shrink pattern in a transformer layer.
5. Describe backpropagation through several layers in words: error signals pass back one layer at a time, and where a unit feeds several units, their contributions add.

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as numpy (`np.`) and scikit-learn, do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_10_demos_helpers_2` draw a plot. That file (and `ch_6_and_10_demos_helpers.py`, which it uses) was written to keep the notebook on ideas, so you never need to open it.
>
> Section 3 needs `mnist.npz`, the same file as Lab 7, in the same folder as this notebook.

## Setup

In [ ]:
import warnings

import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

warnings.simplefilter('ignore', ConvergenceWarning)   # as in Lab 7: we stop some training runs early on purpose

<a id="section-1"></a>

## 1 · Stacking layers

**Each hidden layer treats the previous layer's outputs as its inputs.**

> **A note on symbols.** In the last notebook's Sections 2 to 5, $p$ was the truth and $q$ the model's beliefs, the standard letters in information theory. In this notebook and from now on, we're back to the usual machine-learning convention, as in Lab 7: $p$ (or $\mathbf{p}$, with several classes) is the **network's predicted probability**, and the truth is the label $y$. Watch for this whenever a source compares two or more probability distributions (cross-entropy, KL divergence, the data's distribution versus a model's): check which letter is the truth and which is the model, since many sources write $p$ for the truth.

Here's a small network with two hidden layers of 3 units each, two input features, and one output:

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_two_hidden_layers

plot_two_hidden_layers();

**The forward pass,** extending [the notation from last week](ch_6_and_10_02_neurons_to_networks.ipynb#notation) by one layer. Superscripts count layers: $\mathbf{h}^{(\ell)}$ is hidden layer $\ell$'s outputs, and $W^{(\ell)}$, $\mathbf{b}^{(\ell)}$ are the weights and biases going *into* layer $\ell$. The activation $g$ is ReLU or the sigmoid, and the output layer uses softmax ([last notebook, Section 1](ch_6_and_10_04_softmax_and_cross_entropy.ipynb#section-1)) when there are several classes:

$$
\begin{aligned}
\mathbf{h}^{(1)} &= g\big(W^{(1)} \mathbf{x} + \mathbf{b}^{(1)}\big) && \text{hidden layer 1: from the features} \\
\mathbf{h}^{(2)} &= g\big(W^{(2)} \mathbf{h}^{(1)} + \mathbf{b}^{(2)}\big) && \text{hidden layer 2: from hidden layer 1} \\
\mathbf{p} &= \operatorname{softmax}\big(W^{(3)} \mathbf{h}^{(2)} + \mathbf{b}^{(3)}\big) && \text{output: from hidden layer 2}
\end{aligned}
$$

**Shapes.** $W^{(\ell)}$ has one row per unit in layer $\ell$ and one column per input to it. In the picture, $W^{(1)}$ is $3 \times 2$, $W^{(2)}$ is $3 \times 3$, and $W^{(3)}$ is $1 \times 3$.

Since every layer does the same two things (weighted sum, then activation), the code is a loop.

**Code worth reading.** The forward pass as a loop over layers, in the same `H @ W.T + b` pattern as the last two weeks: one row of `H` per data point. `layers` is a list of `(W, b)` pairs, one per layer, with each `W` shaped (units, inputs). The function returns every hidden layer's outputs (we'll look inside them in Section 3) and the output probabilities.

In [ ]:
def relu(Z):
    return np.maximum(0, Z)


def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def forward(layers, X):
    "Every hidden layer's outputs, and the output probabilities, for the rows of X."
    hidden_outputs = []
    H = X
    for W, b in layers[:-1]:              # every hidden layer: weighted sum, then ReLU
        H = relu(H @ W.T + b)
        hidden_outputs.append(H)
    W_out, b_out = layers[-1]             # the output layer: weighted sum, then softmax
    P = softmax(H @ W_out.T + b_out)
    return hidden_outputs, P


def layers_of(net):
    "A fitted MLPClassifier's (W, b) pairs. scikit-learn stores each W transposed: (inputs, units)."
    return [(W.T, b) for W, b in zip(net.coefs_, net.intercepts_)]

**In scikit-learn,** `hidden_layer_sizes` lists the hidden layers' widths, in order. `hidden_layer_sizes=(64, 2)` means **two** hidden layers: 64 units, then 2. As in Lab 7, `net.coefs_[ℓ]` is stored as (inputs, units), the transpose of our $W^{(\ell)}$, which is why `layers_of` transposes it. (Python counts from 0, so `coefs_[0]` is our $W^{(1)}$.)

**Ask:** A network for MNIST has `hidden_layer_sizes=(64, 2)`: 784 pixel inputs, then 64 units, then 2, then 10 outputs. What are the shapes of $W^{(1)}$, $W^{(2)}$, and $W^{(3)}$, and how many parameters (weights and biases) does it have?


<details>
<summary>Answer</summary>

| | Shape | Weights | Biases |
|---|---|---|---|
| $W^{(1)}$, $\mathbf{b}^{(1)}$ | $64 \times 784$ | 50,176 | 64 |
| $W^{(2)}$, $\mathbf{b}^{(2)}$ | $2 \times 64$ | 128 | 2 |
| $W^{(3)}$, $\mathbf{b}^{(3)}$ | $10 \times 2$ | 20 | 10 |

$$
50{,}176 + 128 + 20 + (64 + 2 + 10) = 50{,}400 \text{ parameters}
$$

Almost all of them are in the first layer, because it connects every pixel to every unit. Section 3 trains this network.

</details>

<a id="why-depth"></a>

**Why stack layers at all?** One hidden layer that is wide enough can fit almost any function. So why do real networks use many layers instead of one huge one?

- **Fewer units for the same job.** "Wide enough" can mean enormous. For many functions, several layers need far fewer units in total than one layer does. (This is a known result; we won't prove it here.)
- **Later layers build on earlier ones.** Each layer computes features of the previous layer's features. In image networks, early layers tend to respond to edges and later ones to strokes and shapes. (We won't show this here.)
- **Layer sizes become a design choice.** With several layers, you choose how many units each one has, and in what order. Sections 2 and 3 show what that choice does: more units give the data room to come apart, and where a narrow layer sits changes what survives it.
- **It's how modern networks are built.** GPT-2 stacks 12 transformer layers, and each contains its own small widen-then-shrink network ([Section 4](#section-4)).

**An honest note:** this notebook's experiments don't show depth beating one big layer; in Section 3, the single 64-unit layer does best. Next week, we'll explore various modern networks, all of which heavily depend upon depth to do amazing things.

<a id="section-2"></a>

## 2 · Widening: room to untangle

Now let's look at what layer **width** does for a neural network.

**The data:** a disk of one class (blue circles) inside a ring of the other (orange triangles), 160 points with two features. No line separates them.

Last week, the XOR network needed 2 hidden units ([h-space](ch_6_and_10_02_neurons_to_networks.ipynb#forward-pass)): each hidden unit drew one line, and in h-space, the output unit separated the classes with one more line.

**Ask:** Will 2 hidden units be enough for a disk inside a ring? Predict before running the next cell.


<details>
<summary>Answer</summary>

No. Each hidden unit contributes one line, and two lines can only cut the plane into wedges and strips. Enclosing the disk takes at least three lines (a triangle). The next cell checks.

</details>

**Code worth reading.** Train networks with 2 and with 3 hidden units, 10 times each from different random starting weights (`random_state`). We use sigmoid hidden units, the `lbfgs` solver (a gradient-based method that works well on small data sets), and no penalty (`alpha=0.0`), and count how many runs classify every training point correctly.

In [ ]:
from ch_6_and_10_demos_helpers import rings_data

X_rings, y_rings = rings_data()
y_rings = (y_rings > 0).astype(int)       # class 1: the disk; class 0: the ring

rings_nets = {}
for h in (2, 3):
    nets = [MLPClassifier(hidden_layer_sizes=(h,), activation='logistic', solver='lbfgs',
                          alpha=0.0, max_iter=5000, random_state=seed).fit(X_rings, y_rings)
            for seed in range(10)]
    accuracies = [net.score(X_rings, y_rings) for net in nets]
    rings_nets[h] = nets[int(np.argmax(accuracies))]           # keep the best run
    perfect = sum(a == 1.0 for a in accuracies)
    print(f'{h} hidden units: {perfect} of 10 runs reach 100%; best run {max(accuracies):.0%}')

Two units never get there; three usually do. The figure shows the best run of each. The dashed lines are the hidden units' own lines, where each unit's score is 0.

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_rings_boundaries

plot_rings_boundaries(rings_nets[2], rings_nets[3], X_rings, y_rings);

**What to notice:**

- **Two lines make a wedge.** The 2-unit network's blue region is bounded by its two dashed lines, and it can't close off the top. Circled points are misclassified.
- **Three lines make a triangle,** which can enclose the disk.

Now look at the 3-unit network the way we looked at XOR: in **h-space**, where each point sits at its hidden outputs $(h_1, h_2, h_3)$. With three hidden units, h-space is 3-D. Drag to rotate it.

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_rings_h_space

plot_rings_h_space(rings_nets[3], X_rings, y_rings)

**Two inputs, three hidden units: the hidden layer moved the data from a 2-D space into a 3-D one.** In the plane, no line separates a disk from a ring. In h-space, a flat plane does: the output unit's plane (gray) has all the disk points on one side. Rotate until you're looking along the plane's edge to see it.

**Giving a layer more units than its inputs gives the data room to come apart.** That's widening, and it's one reason hidden layers often have many units: Lab 7's best MNIST network had 100.

**An honest note:** not every 3-unit run found the triangle. Some got stuck, as in [last week's training runs](ch_6_and_10_03_training_networks.ipynb#section-6): the starting weights matter. A wider layer gives gradient descent more ways downhill, so stuck runs get rarer as the layer widens.

<a id="section-3"></a>

## 3 · Narrowing: a forced summary, and where it sits

In Lab 7, an MNIST network with one hidden layer of **2 units** reached only about 67% test accuracy, and its h-space showed the digits piled on top of each other. Every image had to pass through just two numbers. Two numbers weren't enough. Or were they?

A layer narrower than its input is called a **bottleneck**: everything the network knows about an image has to squeeze through it. Here are four networks. Three of them have a 2-unit bottleneck somewhere.

| `hidden_layer_sizes` | The picture |
|---|---|
| `(2,)` | squeeze immediately (Lab 7's network) |
| `(2, 64)` | squeeze to 2 first, then 64 units |
| `(64, 2)` | 64 units first, then squeeze to 2 |
| `(64,)` | no 2-unit squeeze (for reference) |

(Strictly, every one of these hidden layers is narrower than the 784 pixels. What differs is how *much* each layer throws away, and when.)

**Ask:** Rank the four by test accuracy before running the next cells.

**Code worth reading.** The same data and settings as Lab 7: 10,000 training images chosen at random, pixels scaled to $[0, 1]$, the full 10,000-image test set, and `MLPClassifier`'s defaults (ReLU hidden units, softmax output) with `random_state=0` and `max_iter=100`.

**This cell takes about 30 to 40 seconds.**

In [ ]:
data = np.load('mnist.npz')                                  # the same file as Lab 7
keep = np.random.default_rng(0).choice(60000, size=10000, replace=False)
X_train = data['x_train'][keep].reshape(-1, 784) / 255
y_train = data['y_train'][keep]
X_test = data['x_test'].reshape(-1, 784) / 255
y_test = data['y_test']

mnist_nets = {}
for sizes in [(2,), (2, 64), (64, 2), (64,)]:
    net = MLPClassifier(hidden_layer_sizes=sizes, random_state=0, max_iter=100).fit(X_train, y_train)
    mnist_nets[sizes] = net
    n_params = sum(W.size for W in net.coefs_) + sum(b.size for b in net.intercepts_)
    print(f'{str(sizes):9s} {n_params:7,d} parameters   '
          f'training {net.score(X_train, y_train):.3f}   test {net.score(X_test, y_test):.3f}')

**Reading the results** (Lab 7's `(2,)` network got 0.667; tiny differences between computers are normal):

- **`(2, 64)`, squeeze first, then 64 units, barely beats Lab 7's network** (0.70 vs 0.67), even with 64 more units after the squeeze.
- **`(64, 2)`, 64 units first, then squeeze, reaches 0.85.** Same 2-unit bottleneck, very different result.
- **`(64,)`, no squeeze, reaches 0.95.**

**Code worth reading.** To see why, look inside the two networks at their 2-unit layer. The loop from Section 1 returns every hidden layer's outputs; for `(2,)` the bottleneck is hidden layer 1, and for `(64, 2)` it's hidden layer 2. First, a check that our loop computes the same probabilities as scikit-learn:

In [ ]:
net = mnist_nets[(64, 2)]
hidden, P = forward(layers_of(net), X_test)
print('largest difference from predict_proba:', np.abs(P - net.predict_proba(X_test)).max())
print('hidden layer shapes:', [H.shape for H in hidden])

squeeze_first = forward(layers_of(mnist_nets[(2,)]), X_test)[0][0]    # (2,): hidden layer 1
squeeze_after = hidden[1]                                              # (64, 2): hidden layer 2

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_bottlenecks

plot_bottlenecks(squeeze_first, squeeze_after, y_test,
                 titles=['(2,): the squeeze comes first', '(64, 2): the squeeze comes after 64 units']);

**What to notice:** each point is one test image, placed at the two numbers its bottleneck outputs, and colored by its digit. The circled labels sit at the middle of each digit's points. (The units are ReLU, whose outputs can't be negative, so many points sit right on an axis.)

- **Left, squeeze first:** the two numbers are weighted sums of raw pixels (then ReLU). Most of what distinguishes the digits is thrown away before any layer can use it, and many digits pile up on each other. Widening afterward, as `(2, 64)` did, can't bring that information back: the 64 units only ever see the two numbers.
- **Right, squeeze after a wide layer:** its 64 units first compute features worth keeping, and the 2-unit layer summarizes *those*. The digits spread into far more separate groups.

**A bottleneck forces a summary; what's in the summary depends on what came before it.**

**An honest note:** `(64, 2)` scores 0.98 on training images but 0.85 on test images, so part of its summary is memorized rather than learned. Next week covers how to fight that kind of overfitting in networks.

**Ask:** `(2, 64)` has 64 units after its bottleneck, and `(2,)` has none. Why does `(2, 64)` do only slightly better?


<details>
<summary>Answer</summary>

Because the 64 units only ever see the two numbers that come out of the bottleneck. Whatever the first layer threw away is gone, and no amount of computing on two numbers can recover it. The extra units help a little, by drawing more flexible boundaries in that 2-D space, but they can't add information.

</details>

<a id="section-4"></a>

## 4 · Up and down in modern networks

**The names.** A layer with more units than its input is often called an **up-projection**, and a layer with fewer units than its input a **down-projection**. These are the actual names of the weight matrices in common language-model code (`up_proj`, `down_proj`).

A caution, if you've taken linear algebra: there, a **projection** is a matrix $P$ with $P^2 = P$ (projecting twice changes nothing more). These layers' weight matrices aren't projections in that sense; they're general linear maps, followed by an activation. "Projection" here just means "map to a space of a different size."

**Where you'll meet them.** Every layer of a transformer, the architecture behind large language models, contains a small feed-forward network that does this to each token's vector. In GPT-2, each token is a vector of 768 numbers:

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_transformer_ffn

plot_transformer_ffn();

**Widen to compute, then shrink back to the size the next layer expects.** The up-projection gives the network room to compute (Section 2); the down-projection summarizes the result back into 768 numbers (Section 3). We'll return to transformers on Wednesday, Dec 2.

<a id="backprop-at-depth"></a>

### Backpropagation at depth

Last week, backpropagation found every slope of a one-hidden-layer network ([Section 5](ch_6_and_10_03_training_networks.ipynb#section-5)), starting from the output's error signal $p - y$. With softmax and cross-entropy, it starts from the same pattern, $\mathbf{p} - \mathbf{y}$, one entry per class ([last notebook, Section 5](ch_6_and_10_04_softmax_and_cross_entropy.ipynb#section-5)). With more layers, nothing new is needed, with one picture's worth of care: **a hidden unit usually feeds several units in the next layer**, so it receives error from each of them.

In [ ]:
from ch_6_and_10_demos_helpers import plot_error_sum

plot_error_sum();

**In words:**

1. **Where a unit feeds several units, their error contributions add.** Each edge carries back the next unit's error signal times the edge's weight, and the hidden unit adds them up, then multiplies by its own activation's slope (the picture uses a sigmoid unit, whose slope is $h_k(1 - h_k)$).
2. **Each layer's error signals are computed from the layer above.** So the backward pass is the forward pass in reverse: start at the output with $\mathbf{p} - \mathbf{y}$, and step back one layer at a time, all the way to the first.

Libraries such as PyTorch do exactly this automatically, for any stack of layers. The optional appendix at the end of this notebook has the derivation.

<a id="summary"></a>

## 5 · Summary

- **Stacking layers:** each hidden layer treats the previous layer's outputs as its inputs, so the forward pass is a loop over layers. `hidden_layer_sizes=(64, 2)` means two hidden layers.
- **Widening** a layer (more units than its input) gives h-space more dimensions, so data that no line separates in the original features can become separable by a plane.
- **Narrowing** a layer (a **bottleneck**) forces a summary. The same 2-unit bottleneck reaches 0.85 test accuracy after a 64-unit layer, but 0.67 to 0.70 when it comes first: information thrown away early can't be recovered later.
- **Up- and down-projections** are the names for widening and narrowing layers. Every transformer layer widens 4× and shrinks back.
- **Backpropagation at depth** passes error signals back one layer at a time, adding the contributions where a unit feeds several units.

**Today, both notebooks together:** a network's **output layer** is softmax, scored by cross-entropy; its **hidden layers** widen and narrow the representation on the way there.

**Next week:** words become vectors, and a network predicts the next word, trained with today's loss.

# Practice Questions

These questions cover the output layer of a deeper network, widening, bottlenecks and where they sit, and backpropagation through several layers. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

For reference:

- **Reading `hidden_layer_sizes`:** it lists only the hidden layers' widths, in order. For MNIST, every network starts with 784 pixel inputs and ends with 10 outputs, so `(64, 2)` is 784 → 64 → 2 → 10.
- **Section 3's MNIST test accuracies:** `(2,)` 0.67, `(2, 64)` 0.70, `(64, 2)` 0.85, `(64,)` 0.95.

### Question 1: Why softmax on the output layer?

A network classifies MNIST digits. Its last hidden layer feeds 10 output units, and each output unit computes a score $z_k$, which can be any number.

(a) To turn the 10 scores into the network's predicted probabilities, which should the output layer apply?

- A. ReLU to each score.
- B. A sigmoid to each score.
- C. Softmax to all 10 scores together.
- D. Nothing: use the scores as they are.

(b) The hidden layers use ReLU. Why is ReLU fine there but not at the output?

(See [Section 1](#section-1), and [last notebook's output layer](ch_6_and_10_04_softmax_and_cross_entropy.ipynb#output-layer).)

<details>
<summary>Hint</summary>

List what 10 numbers need in order to be probabilities, then check each option against the list.

</details>

<details>
<summary>Answer</summary>

**(a) C, softmax.** Probabilities must each be between 0 and 1 and sum to 1, with a bigger score giving a bigger probability. Softmax does all three.

- **A (ReLU)** can give values above 1, or all zeros, and they don't sum to 1.
- **B (a sigmoid on each)** keeps each value between 0 and 1, but the 10 values don't have to sum to 1: the network could say 0.9 to both "3" and "8".
- **D (raw scores)** can be negative or bigger than 1.

**(b) Hidden outputs aren't probabilities.** They're features passed to the next layer, so they don't need to be between 0 and 1 or sum to anything. Only the output layer has to produce probabilities.

</details>

### Question 2: Hidden units as lines

A network has 2 input features and one hidden layer of sigmoid units. Each hidden unit draws one line in the plane of the features. The data are a disk of one class inside a ring of the other.

What's the fewest hidden units that can enclose the disk, and why don't fewer work?

(See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

**3 hidden units.** Three lines can form a triangle around the disk. With 2 lines, the network can only cut the plane into wedges and strips, and none of those encloses the disk: in Section 2, no 2-unit run classified every point correctly, and the best reached 85%.

</details>

### Question 3: Widening, true or false

Section 2's network has 2 input features, 3 hidden units, and one output unit. Mark each statement true or false.

(a) Its h-space, where each point sits at its hidden outputs $(h_1, h_2, h_3)$, is 3-dimensional.

(b) In h-space, the output unit separates the disk from the ring with a flat plane.

(c) Widening to 3 hidden units adds new information about each data point that wasn't in its 2 features.

(d) Two training runs of the same 3-unit network, from different starting weights, can end differently: one might find the triangle and another one might not.

(See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

**(a) True.** One dimension per hidden unit.

**(b) True.** No line separates the classes in the original plane, but in the 3-D h-space a plane does.

**(c) False.** Every hidden output is computed from the same 2 features, so there's no new information. Widening *rearranges* the data in a space with more dimensions, which gives the classes room to come apart.

**(d) True.** Some 3-unit runs got stuck. The starting weights matter, and wider layers make stuck runs rarer.

</details>

### Question 4: Where should the bottleneck go?

Three MNIST networks, each with the usual 784 pixel inputs and 10 outputs:

| `hidden_layer_sizes` | The layers |
|---|---|
| `(3,)` | 784 → **3** → 10: one hidden layer of 3 units |
| `(3, 128)` | 784 → **3** → 128 → 10: 3 units, then 128 |
| `(128, 3)` | 784 → 128 → **3** → 10: 128 units, then 3 |

(a) Without training anything, rank them from the highest expected test accuracy to the lowest.

(b) `(128, 3)` and `(3, 128)` have the same layers in a different order. Explain in one or two sentences why their order matters.

(See [Section 3](#section-3).)

<details>
<summary>Hint</summary>

All three have a 3-unit bottleneck. For each, ask what the bottleneck gets to summarize: raw pixels, or features a wider layer already computed?

</details>

<details>
<summary>Answer</summary>

**(a) `(128, 3)`, then `(3, 128)`, then `(3,)`.** Trained like Section 3's networks, they reach about 0.92, 0.80, and 0.75.

- **`(128, 3)` is best:** its 3-unit layer summarizes 128 features the first layer already computed, so less of what matters is lost.
- **`(3, 128)` is next:** its 3-unit layer squeezes the raw pixels, so most of the information is lost right away. But the 128 units after it can still draw more flexible boundaries among those 3 numbers, which helps some.
- **`(3,)` is last:** the same early squeeze, with nothing after it but the output layer.

**(b) What the bottleneck summarizes.** In `(128, 3)`, the 128 units first compute useful features, and the 3-unit layer summarizes those. In `(3, 128)`, the 3-unit layer squeezes the raw pixels right away, and the 128 units after it only ever see 3 numbers, so they can't recover what was thrown away.

</details>

### Question 5: Why don't more units after the squeeze help much?

`(2, 64)` has 64 units after its 2-unit layer and `(2,)` has none, yet `(2, 64)` reaches only 0.70 test accuracy against 0.67. Which explanation is right?

- A. The 64 units only see the two numbers that come out of the 2-unit layer, so information the first layer threw away can't be recovered.
- B. 64 units are so many that the network overfits.
- C. ReLU units can't follow a 2-unit layer.
- D. Adding units after a narrow layer can never help.

(See [Section 3](#section-3).)

<details>
<summary>Answer</summary>

**A.** Whatever the first layer discarded is gone; the 64 units can only compute from two numbers.

- **B is wrong:** `(2, 64)` scores only 0.73 on its *training* images too, so it isn't memorizing; it's missing information.
- **C is wrong:** nothing stops a ReLU layer from following any layer.
- **D is wrong:** the 64 units *did* help a little (0.67 to 0.70), by drawing more flexible boundaries in the 2-D space. And some networks widen after a narrow layer on purpose, for example to rebuild an image from a compact summary. What extra units can't do is add back information that's already gone.

</details>

### Question 6: Reading the bottleneck plot

Network A and Network B are Section 3's `(2, 64)` and `(64, 2)`, in some order. The figure shows each network's **2-unit layer** for the test images, colored by digit, with each digit's label circled at the middle of its points. (Run Section 3's cells first.)

(a) Which network is `(2, 64)` (the 2-unit layer comes first), and which is `(64, 2)` (a 64-unit layer comes first)? How can you tell from the picture?

(b) Explain why, using the terms **down-projection** (a layer with fewer units than its input) and **up-projection** (a layer with more units than its input).

(See [Section 3](#section-3).)

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_bottleneck_question

plot_bottleneck_question(mnist_nets, X_test, y_test);

<details>
<summary>Hint</summary>

A 2-unit layer that summarizes raw pixels keeps much less than one that summarizes 64 computed features. Which panel looks like it kept more?

</details>

<details>
<summary>Answer</summary>

**(a) Network A is `(64, 2)`, and Network B is `(2, 64)`.** In A, the digits spread into far more separate groups; in B, many digits pile on top of each other, as in Lab 7's `(2,)` network.

**(b)** Both networks down-project from 784 pixels to 2 numbers eventually, but differently:

- **`(64, 2)` down-projects in two steps.** 784 → 64 keeps enough room for the first layer to compute useful features, and 64 → 2 then summarizes those features.
- **`(2, 64)` down-projects all at once,** 784 → 2, straight from raw pixels, and most of the information is lost there. Its up-projection, 2 → 64, comes too late: it can only compute from the two numbers it's given.

</details>

### Question 7: Backpropagation and depth

Answer each in two or three sentences.

(a) How do error signals get from the output back to the first hidden layer? What happens at a hidden unit that feeds several units in the next layer?

(b) A network with one hidden layer can fit almost any function if that layer is wide enough. So why do real networks use several hidden layers?

(See [Backpropagation at depth](#backprop-at-depth) and [Why stack layers at all?](#why-depth))

<details>
<summary>Answer</summary>

**(a) One layer at a time, backward.** The backward pass starts at the output's error signal, $\mathbf{p} - \mathbf{y}$, and computes each layer's error signals from the layer above it, all the way to the first. At a unit that feeds several units, each edge carries back the next unit's error signal times the edge's weight, and the **contributions add**; the sum is then multiplied by the unit's own activation slope.

**(b) Because "wide enough" can mean enormous.** One hidden layer may need a huge number of units to fit a complicated function. With several layers, later layers build on what earlier layers computed, which usually takes far fewer units in total.

</details>

<a id="appendix"></a>

## Appendix (optional): where a unit's error signal comes from

This derives the rule in [Backpropagation at depth](#backprop-at-depth), using the notation of [last week's Section 5](ch_6_and_10_03_training_networks.ipynb#section-5). A unit's **error signal** is the slope of the loss with respect to that unit's score, $\delta = \partial L / \partial z$. For the output, it's $\mathbf{p} - \mathbf{y}$.

Take hidden unit $k$, with score $z^{(1)}_k$ and output $h_k = g\big(z^{(1)}_k\big)$. It feeds units $j = 1, \dots, J$ in the next layer, one edge each, with weight $w^{(2)}_{jk}$ (into unit $j$, from unit $k$). Each of those units computes its score from all the hidden outputs:

$$
z^{(2)}_j = \sum_{m} w^{(2)}_{jm}\, h_m + b^{(2)}_j
$$

The only term containing $h_k$ is $w^{(2)}_{jk}\, h_k$, so

$$
\frac{\partial z^{(2)}_j}{\partial h_k} = w^{(2)}_{jk}
$$

So $h_k$ affects the loss along several paths, one through each $z^{(2)}_j$. For a function of several variables, each of which depends on $h_k$, the **multivariable chain rule** adds one term per path:

$$
\frac{\partial L}{\partial h_k} = \sum_{j=1}^{J} \frac{\partial L}{\partial z^{(2)}_j} \cdot \frac{\partial z^{(2)}_j}{\partial h_k}
$$

Nudging $h_k$ nudges every $z^{(2)}_j$, and the changes in the loss add up. Now the derivation:

$$
\begin{aligned}
\delta^{(1)}_k &= \frac{\partial L}{\partial z^{(1)}_k} && \text{definition of the error signal} \\
&= \frac{\partial L}{\partial h_k} \cdot \frac{\partial h_k}{\partial z^{(1)}_k} && \text{chain rule: } z^{(1)}_k \text{ affects } L \text{ only through } h_k \\
&= \Big(\sum_{j=1}^{J} \frac{\partial L}{\partial z^{(2)}_j} \cdot \frac{\partial z^{(2)}_j}{\partial h_k}\Big)\, g'\big(z^{(1)}_k\big) && \text{multivariable chain rule} \\
&= \Big(\sum_{j=1}^{J} \delta^{(2)}_j\, w^{(2)}_{jk}\Big)\, g'\big(z^{(1)}_k\big) && \text{definition of } \delta^{(2)}_j\text{; } \partial z^{(2)}_j / \partial h_k = w^{(2)}_{jk}
\end{aligned}
$$

In the picture, pieces 1, 2, 3 are the terms of the sum, and piece 4 is the activation's slope: for a sigmoid, $g'(z) = h_k(1 - h_k)$, as in the picture; for ReLU, it's 1 when $z > 0$ and 0 otherwise. With one output unit, the sum has one term, and this is last week's $\delta^{(1)}_k = \delta^{(2)}\, w^{(2)}_k\, h_k(1 - h_k)$.

**Every layer works the same way.** For any hidden layer $\ell$:

$$
\delta^{(\ell)}_k = \Big(\sum_{j} \delta^{(\ell + 1)}_j\, w^{(\ell + 1)}_{jk}\Big)\, g'\big(z^{(\ell)}_k\big)
$$

and each weight's slope is (the error signal at the unit it feeds) $\times$ (the input it multiplies): $\partial L / \partial w^{(\ell)}_{kj} = \delta^{(\ell)}_k\, h^{(\ell - 1)}_j$, with $\mathbf{h}^{(0)} = \mathbf{x}$. In code, with one row per data point, the sum over $j$ is a matrix product: `D = (D_above @ W_above) * slope`.